# Rapport — partie 2 : détection de la négation

**Binôme** : AMARA Yanis, TOMAS Alban · Projet de traitement automatique du langage en santé (ESIEE, E5-BIO)

Une fois une maladie repérée dans un texte (partie 1), il faut savoir si le texte **affirme** sa présence ou la
**nie** : « no fever » ne dit pas que le patient a de la fièvre. Ce notebook présente notre méthode à base de règles,
la construction d'un petit jeu de test annoté à deux, l'évaluation et l'analyse des erreurs, dans l'ordre où nous
avons travaillé.

Tout est **recalculé** à l'exécution (quelques secondes, sans GPU), à partir des règles (`src/negation.py`), des
20 phrases fictives (`data/negation.jsonl`) et de nos annotations : feuilles des deux membres, décisions communes et
empreinte des règles figées (`resultats/negation/`). Historique daté : `RAPPORT.md`, section 5.

In [1]:
import sys, json, hashlib
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.negation import REGLES, detecter

NEGATION = RACINE / "resultats" / "negation"
LABELS = ["nie", "non_nie"]
pd.set_option("display.max_colwidth", 100)

## 1. La tâche et sa règle de jeu

La **mention cible est donnée** (phrase + positions) : la méthode n'a qu'à décider si elle est `nie` (le texte dit
qu'elle est absente) ou `non_nie` (tous les autres cas : affirmée, incertaine, hypothétique). « Non nié » ne veut pas
dire « confirmé » : « lupus cannot be excluded » est `non_nie`, car le lupus reste possible.

Le livret impose un ordre qui protège l'évaluation : **écrire les règles avant de voir les phrases de test**, puis
générer 20 phrases fictives, les annoter à deux, les séparer en développement / test, ajuster sur le développement
seulement, et évaluer une seule fois sur le test.

## 2. Étape 1 : les règles, figées avant les phrases

Nous avons adapté l'algorithme **NegEx** (Chapman et al., 2001), vu en cours, en anglais (langue de BC5CDR) :

1. **Déclencheurs avant** la mention (« no », « not », « without », « negative for », « no evidence of »…) : la
   négation porte vers la droite ;
2. **Déclencheurs après** (« was ruled out », « was excluded », « absent ») : elle porte vers la gauche ;
3. **Fausses négations** (« no doubt », « not only », « no increase », « cannot be excluded ») : leurs mots ne
   déclenchent rien ;
4. **Portée limitée** : elle s'arrête à une **terminaison** (« but », « however », « ; », « . »…) et au plus à
   **8 unités** (mots ou signes de ponctuation) ;
5. chaque décision garde la règle et le déclencheur utilisés, pour pouvoir l'expliquer.

Cette version `v0` a été figée le 07/10/2026 (empreinte enregistrée) **avant** la génération des phrases. On vérifie
que le code n'a pas changé depuis :

In [2]:
gel = json.loads((NEGATION / "regles_v0.json").read_text(encoding="utf-8"))
empreinte = hashlib.sha256(json.dumps(REGLES["v0"], ensure_ascii=False, sort_keys=True).encode()).hexdigest()
print("Figée le", gel["date_gel"], "—", gel["note"], "; code inchangé depuis :", empreinte == gel["sha256"])
pd.DataFrame({cle: [len(REGLES["v0"][cle]), ", ".join(REGLES["v0"][cle][:8]) + " …"]
              for cle in ("pre", "post", "pseudo", "terminaisons")}, index=["nombre", "exemples"]).T

Figée le 2026-10-07T16:40:40 — figée avant la génération des 20 phrases fictives ; code inchangé depuis : True


,nombre,exemples
pre,19,"no, not, without, denies, denied, deny, neither, nor …"
post,26,"was ruled out, were ruled out, is ruled out, has been ruled out, have been ruled out, was exclud..."
pseudo,22,"no doubt, without doubt, not only, no increase, no change, no significant change, no further, no..."
terminaisons,20,"but, however, although, though, yet, except, which, who …"


Avant d'écrire la moindre phrase de test, nous avons vérifié le comportement sur les exemples du cours :

In [3]:
def juger(phrase, mention):
    i = phrase.index(mention)
    d = detecter(phrase, i, i + len(mention), "v0")
    return {"phrase": phrase, "mention": mention, "décision": d.label, "règle": d.regle, "déclencheur": d.declencheur}

pd.DataFrame([juger(p, m) for p, ms in [
    ("No fever nor chills, but a persistent cough. Headache absent.", ["fever", "chills", "cough", "Headache"]),
    ("There is no doubt: an infection is present.", ["infection"]),
    ("He presents not only a fever but also a cough.", ["fever", "cough"]),
    ("An embolism cannot be excluded.", ["embolism"]),
    ("Appendicitis is not ruled out.", ["Appendicitis"]),
] for m in ms])

,phrase,mention,décision,règle,déclencheur
0,"No fever nor chills, but a persistent cough. Headache absent.",fever,nie,pre,no
1,"No fever nor chills, but a persistent cough. Headache absent.",chills,nie,pre,no
2,"No fever nor chills, but a persistent cough. Headache absent.",cough,non_nie,NaN,NaN
3,"No fever nor chills, but a persistent cough. Headache absent.",Headache,nie,post,absent
4,There is no doubt: an infection is present.,infection,non_nie,NaN,NaN
5,He presents not only a fever but also a cough.,fever,non_nie,NaN,NaN
6,He presents not only a fever but also a cough.,cough,non_nie,NaN,NaN
7,An embolism cannot be excluded.,embolism,non_nie,NaN,NaN
8,Appendicitis is not ruled out.,Appendicitis,non_nie,NaN,NaN


Les pièges classiques (portée fermée par « but », négation placée après, « no doubt », « not only »,
« cannot be excluded ») sont bien traités.

## 3. Étape 2 : les 20 phrases fictives

Générées le 07/10/2026 avec l'aide d'un LLM (Claude), **après** le gel des règles. Pour tester des cas variés, elles
sont organisées en **10 familles** de deux phrases proches (même tournure, l'une niée, l'autre non) : négation simple,
portée fermée par « but », coordination « neither… nor », liste, négation après la mention, « cannot be excluded »,
fausse négation, double négation, hypothèse, portée longue.

In [4]:
phrases = pd.DataFrame([json.loads(l) for l in open(RACINE / "data" / "negation.jsonl", encoding="utf-8")])
assert all(r.phrase[r.positions[0]:r.positions[1]] == r.mention for r in phrases.itertuples())   # positions justes
phrases[["phrase_id", "family_id", "famille", "phrase", "mention"]]

,phrase_id,family_id,famille,phrase,mention
0,NEG001,F01,negation_simple,The patient reported no headache after the first infusion.,headache
1,NEG002,F01,negation_simple,The patient reported a severe headache after the first infusion.,headache
2,NEG003,F02,portee_mais,"No fever was recorded, but the child developed a persistent cough.",fever
3,NEG004,F02,portee_mais,"No fever was recorded, but the child developed a persistent cough.",persistent cough
4,NEG005,F03,coordination_ni,She experienced neither nausea nor vomiting during the treatment.,vomiting
5,NEG006,F03,coordination_ni,She experienced nausea and vomiting during the treatment.,vomiting
6,NEG007,F04,coordination_liste,"Examination revealed no rash, jaundice or edema, and mild hypertension was noted.",edema
7,NEG008,F04,coordination_liste,"Examination revealed no rash, jaundice or edema, and mild hypertension was noted.",hypertension
8,NEG009,F05,negation_post,Acute hepatitis was ruled out by liver biopsy.,Acute hepatitis
9,NEG010,F05,negation_post,Acute hepatitis was confirmed by liver biopsy.,Acute hepatitis


## 4. Étape 3 : double annotation

Nous avons écrit un **guide d'annotation d'une page** (`resultats/negation/guide_annotation.md`) : définition des deux
labels, règles de portée, faux amis, exemples. Chacun a ensuite annoté les 20 mentions **seul**, sans voir l'autre
feuille ni le label proposé à la génération. Accord mesuré par le **kappa de Cohen**, qui corrige l'accord observé de
la part due au hasard (1 = accord parfait, 0 = pas mieux que le hasard) :

In [5]:
def lire(nom):
    return pd.read_csv(NEGATION / f"annotation_{nom}.csv", dtype=str).set_index("phrase_id")["label"].str.strip()

a, t = lire("AMARA"), lire("TOMAS")
accord = (a == t).mean()
hasard = sum((a == l).mean() * (t == l).mean() for l in LABELS)
kappa = (accord - hasard) / (1 - hasard)
print(f"Accord observé : {accord:.2f} ; accord attendu par hasard : {hasard:.2f} ; kappa de Cohen : {kappa:.2f} ; "
      f"désaccords : {(a != t).sum()} sur {len(a)}")
pd.crosstab(a.rename("AMARA"), t.rename("TOMAS"))

Accord observé : 0.75 ; accord attendu par hasard : 0.49 ; kappa de Cohen : 0.51 ; désaccords : 5 sur 20


TOMAS,nie,non_nie
AMARA,,
nie,8,1
non_nie,4,7


Un kappa de 0,51 est un accord **modéré** : la tâche paraît simple, mais les cas limites sont nombreux. Les
désaccords ont été discutés ensemble et tranchés à l'aide du guide :

In [6]:
desaccords = pd.read_csv(NEGATION / "desaccords_annotation.csv", dtype=str).set_index("phrase_id")
desaccords[["phrase", "mention", "label_AMARA", "label_TOMAS", "label_final", "resolution"]]

,phrase,mention,label_AMARA,label_TOMAS,label_final,resolution
phrase_id,,,,,,
NEG007,"Examination revealed no rash, jaundice or edema, and mild hypertension was noted.",edema,nie,non_nie,nie,presence de negation
NEG008,"Examination revealed no rash, jaundice or edema, and mild hypertension was noted.",hypertension,non_nie,nie,non_nie,portee fermee par « and »
NEG010,Acute hepatitis was confirmed by liver biopsy.,Acute hepatitis,non_nie,nie,non_nie,absence de negation
NEG013,There was no increase in seizure frequency after the dose change.,seizure,non_nie,nie,non_nie,variation qui est niee
NEG018,Nerve conduction studies showed no evidence of neuropathy.,neuropathy,non_nie,nie,nie,« no evidence of »


Ce que ces désaccords nous ont appris :
- **NEG007 / NEG008** (liste « no rash, jaundice or edema, **and** mild hypertension was noted ») : jusqu'où va la
  négation dans une liste ? « or » la prolonge (edema est nié), « and … was noted » ouvre une nouvelle affirmation
  (hypertension n'est pas niée).
- **NEG013** (« no increase in seizure frequency ») : le mot « no » est présent, mais c'est la *hausse* qui est niée ;
  les crises existent. C'est exactement la « fausse négation » de nos règles.
- **NEG018** (« no evidence of neuropathy ») : formule standard de négation en médecine.
- **NEG010** (« was confirmed ») : aucun mot de négation ; le désaccord venait d'une inversion de label, repérée grâce
  à la justification écrite à côté.

La **référence finale** réunit les 15 accords et les 5 décisions communes. Elle coïncide avec les labels proposés à
la génération (vérifié ci-dessous), mais elle est désormais justifiée par deux annotateurs.

In [7]:
reference = a.where(a == t, desaccords["label_final"].reindex(a.index))
assert reference.isin(LABELS).all()
phrases["reference"] = phrases.phrase_id.map(reference)
print("Référence identique aux labels de génération :", (phrases.reference == phrases.label).all())

Référence identique aux labels de génération : True


## 5. Étape 4 : développement et test

Les deux phrases d'une même famille vont **dans le même ensemble** : sinon, la phrase de test aurait sa « jumelle »
dans le développement et le test serait trop facile. Résultat : 5 familles de chaque côté, 5 mentions niées et 5 non
niées par ensemble.

In [8]:
assert (phrases.groupby("family_id").partition.nunique() == 1).all()
display(pd.crosstab(phrases.partition, phrases.reference))
phrases.drop_duplicates("family_id").groupby("partition").famille.apply(", ".join).to_frame()

reference,nie,non_nie
partition,,
dev,5,5
test,5,5


,famille
partition,
dev,"negation_simple, portee_mais, coordination_liste, incertitude_exclure, hypothese_condition"
test,"coordination_ni, negation_post, pseudo_negation, double_negation, portee_longue"


## 6. Étape 5 : réglage sur le développement

On applique les règles v0 aux 10 phrases de développement, avec la règle utilisée pour chaque décision :

In [9]:
def appliquer(df, version="v0", **modifs):
    regles = {**REGLES[version], **modifs}
    REGLES["essai"] = regles
    decisions = [detecter(r.phrase, r.positions[0], r.positions[1], "essai") for r in df.itertuples()]
    return df.assign(predit=[d.label for d in decisions], regle=[d.regle for d in decisions],
                     declencheur=[d.declencheur for d in decisions])

dev = appliquer(phrases[phrases.partition == "dev"])
print(f"Développement : {(dev.predit == dev.reference).sum()} / {len(dev)} correctes")
dev[["phrase_id", "phrase", "mention", "reference", "predit", "regle", "declencheur"]]

Développement : 10 / 10 correctes


,phrase_id,phrase,mention,reference,predit,regle,declencheur
0,NEG001,The patient reported no headache after the first infusion.,headache,nie,nie,pre,no
1,NEG002,The patient reported a severe headache after the first infusion.,headache,non_nie,non_nie,NaN,NaN
2,NEG003,"No fever was recorded, but the child developed a persistent cough.",fever,nie,nie,pre,no
3,NEG004,"No fever was recorded, but the child developed a persistent cough.",persistent cough,non_nie,non_nie,NaN,NaN
6,NEG007,"Examination revealed no rash, jaundice or edema, and mild hypertension was noted.",edema,nie,nie,pre,no
7,NEG008,"Examination revealed no rash, jaundice or edema, and mild hypertension was noted.",hypertension,non_nie,non_nie,NaN,NaN
10,NEG011,Drug-induced lupus cannot be excluded at this stage.,Drug-induced lupus,non_nie,non_nie,NaN,NaN
11,NEG012,Drug-induced lupus was excluded after serological testing.,Drug-induced lupus,nie,nie,post,was excluded
16,NEG017,"If neuropathy appears, the dose of cisplatin should be reduced.",neuropathy,non_nie,non_nie,NaN,NaN
17,NEG018,Nerve conduction studies showed no evidence of neuropathy.,neuropathy,nie,nie,pre,no evidence of


Les 10 phrases de développement sont correctement classées : **nous n'avons rien modifié** et la version v0 est
celle évaluée sur le test. Changer une règle sans erreur à corriger aurait seulement ajusté les règles à nos propres
phrases.

**À quoi sert chaque composant ?** Pour comprendre les règles (et non pour les régler), on retire un composant à la
fois et on regarde ce qui casse, sur le développement et sur les exemples du cours (jamais sur le test) :

In [10]:
COURS = [("No fever nor chills, but a persistent cough. Headache absent.",
          {"fever": "nie", "chills": "nie", "cough": "non_nie", "Headache": "nie"}),
         ("There is no doubt: an infection is present.", {"infection": "non_nie"}),
         ("He presents not only a fever but also a cough.", {"fever": "non_nie", "cough": "non_nie"}),
         ("An embolism cannot be excluded.", {"embolism": "non_nie"}),
         ("No chest pain, but dyspnea persists.", {"chest pain": "nie", "dyspnea": "non_nie"}),
         ("Appendicitis is not ruled out.", {"Appendicitis": "non_nie"})]
cours = pd.DataFrame([{"phrase_id": "cours", "phrase": ph, "mention": m, "positions": [ph.index(m), ph.index(m) + len(m)],
                       "reference": l} for ph, ms in COURS for m, l in ms.items()])
dev_ids = pd.concat([phrases[phrases.partition == "dev"], cours], ignore_index=True)
variantes = {"v0 complète": {}, "sans fausses négations": {"pseudo": []}, "sans terminaisons": {"terminaisons": []},
             "portée 4 mots": {"portee_max": 4}, "portée illimitée (50 mots)": {"portee_max": 50}}
lignes = []
for nom, modifs in variantes.items():
    r = appliquer(dev_ids, **modifs)
    fausses = r[r.predit != r.reference]
    lignes.append({"variante": nom, f"correctes (/{len(r)})": int((r.predit == r.reference).sum()),
                   "phrases qui cassent": ", ".join(f"{x.phrase_id} ({x.mention})" for x in fausses.itertuples()) or "—"})
pd.DataFrame(lignes).set_index("variante")

,correctes (/21),phrases qui cassent
variante,,
v0 complète,21,—
sans fausses négations,20,cours (fever)
sans terminaisons,19,"cours (cough), cours (dyspnea)"
portée 4 mots,20,NEG007 (edema)
portée illimitée (50 mots),20,NEG008 (hypertension)


Chaque composant protège au moins un cas : sans fausses négations, « not only a fever » devient une négation ; sans
terminaisons, « but » ne ferme plus la portée (« cough », « dyspnea » deviennent niés). Sur nos phrases, NEG004
(« No fever …, but … persistent cough ») reste juste même sans terminaisons, mais seulement parce que la mention est
loin du « No » : la limite de portée sert de filet de sécurité. Cette limite est un compromis : à 4 mots, la négation
n'atteint plus le dernier élément de la liste (NEG007) ; sans limite, elle déborde sur « hypertension » (NEG008).
Nous y reviendrons à propos d'une erreur du test.

## 7. Évaluation unique sur le test

In [11]:
test = appliquer(phrases[phrases.partition == "test"])
matrice = pd.crosstab(pd.Categorical(test.reference, LABELS), pd.Categorical(test.predit, LABELS),
                      rownames=["référence"], colnames=["prédit"], dropna=False)
display(matrice)

lignes = []
for c in LABELS:
    vp = ((test.predit == c) & (test.reference == c)).sum()
    fp = ((test.predit == c) & (test.reference != c)).sum()
    fn = ((test.predit != c) & (test.reference == c)).sum()
    p, r = vp / (vp + fp), vp / (vp + fn)
    lignes.append({"classe": c, "VP": vp, "FP": fp, "FN": fn, "précision": round(p, 2), "rappel": round(r, 2),
                   "F1": round(2 * p * r / (p + r), 2)})
scores = pd.DataFrame(lignes).set_index("classe")
display(scores)
print(f"Moyenne des deux F1 : {scores.F1.mean():.2f}")

prédit,nie,non_nie
référence,,
nie,4,1
non_nie,1,4


,VP,FP,FN,précision,rappel,F1
classe,,,,,,
nie,4,1,1,0.8,0.8,0.8
non_nie,4,1,1,0.8,0.8,0.8


Moyenne des deux F1 : 0.80


In [12]:
test[["phrase_id", "famille", "phrase", "mention", "reference", "predit", "regle", "declencheur"]]

,phrase_id,famille,phrase,mention,reference,predit,regle,declencheur
4,NEG005,coordination_ni,She experienced neither nausea nor vomiting during the treatment.,vomiting,nie,nie,pre,neither
5,NEG006,coordination_ni,She experienced nausea and vomiting during the treatment.,vomiting,non_nie,non_nie,NaN,NaN
8,NEG009,negation_post,Acute hepatitis was ruled out by liver biopsy.,Acute hepatitis,nie,nie,post,was ruled out
9,NEG010,negation_post,Acute hepatitis was confirmed by liver biopsy.,Acute hepatitis,non_nie,non_nie,NaN,NaN
12,NEG013,pseudo_negation,There was no increase in seizure frequency after the dose change.,seizure,non_nie,non_nie,NaN,NaN
13,NEG014,pseudo_negation,No seizures occurred after the dose change.,seizures,nie,nie,pre,no
14,NEG015,double_negation,The normal ECG did not exclude myocardial ischemia.,myocardial ischemia,non_nie,nie,pre,not
15,NEG016,double_negation,Myocardial ischemia was absent on the stress test.,Myocardial ischemia,nie,nie,post,was absent
18,NEG019,portee_longue,"The patient did not, despite the high cumulative dose of doxorubicin, develop cardiomyopathy.",cardiomyopathy,nie,non_nie,NaN,NaN
19,NEG020,portee_longue,Cardiomyopathy developed despite the low cumulative dose of doxorubicin.,Cardiomyopathy,non_nie,non_nie,NaN,NaN


**Résultat : 8 mentions sur 10 correctes, F1 = 0,80 pour les deux classes, moyenne des deux F1 = 0,80.** Une erreur
dans chaque sens. Sur 10 exemples, une seule erreur fait varier le F1 d'environ 0,1 : ce score décrit le comportement
des règles sur des cas difficiles choisis, pas une performance sur de vrais dossiers.

Les cas difficiles réussis : la coordination « neither nausea nor vomiting », la négation placée après
(« was ruled out »), la fausse négation « no increase in seizure frequency » (NEG013, justement discutée entre nous),
et l'absence placée après (« was absent »).

## 8. Analyse des deux erreurs

**NEG015 — double négation** (référence `non_nie`, prédit `nie`)

> The normal ECG did **not** exclude **myocardial ischemia**.

« not » est reconnu comme déclencheur et « ischemia » est dans sa portée. Mais « not » porte sur le verbe « exclude » :
ne pas exclure une ischémie, c'est la laisser **possible**. Nos règles connaissent « cannot be excluded » et
« not ruled out » comme fausses négations, mais pas la forme active « did not exclude ». C'est le piège décrit en
cours : appliquer « contient *not*, donc absent » inverse ici l'information, ce qui est grave en santé (une ischémie
possible serait classée comme écartée).

**NEG019 — portée longue** (référence `nie`, prédit `non_nie`)

> The patient did **not**, despite the high cumulative dose of doxorubicin, develop **cardiomyopathy**.

« not » est bien un déclencheur, mais 10 unités (mots et virgules) le séparent de « cardiomyopathy », au-delà de la
portée maximale de 8. Cette limite évite d'étendre une négation à toute une phrase (on l'a vu avec NEG008 sur le développement) ;
ici, elle coupe une négation réelle séparée de sa mention par une incise.

In [13]:
from src.negation import _tokens   # même découpage que les règles : mots et ponctuation
r = phrases.set_index("phrase_id").loc["NEG019"]
unites = [u for u, debut, _ in _tokens(r.phrase)]
i_not, i_mention = unites.index("not"), unites.index("cardiomyopathy")
print(unites[i_not + 1:i_mention], "→", i_mention - i_not - 1, "unités entre « not » et la mention ; portée maximale :",
      REGLES["v0"]["portee_max"])

[',', 'despite', 'the', 'high', 'cumulative', 'dose', 'of', 'doxorubicin', ',', 'develop'] → 10 unités entre « not » et la mention ; portée maximale : 8


**Corrections possibles, non appliquées** : ajouter « did not exclude », « does not rule out » aux fausses négations
(NEG015) ; calculer la portée sur la structure de la phrase (le complément du verbe nié) plutôt qu'en nombre de mots
(NEG019). Les appliquer maintenant reviendrait à régler les règles sur le test : le score rapporté reste celui de v0.

## 9. Limites et conclusion

- **Taille** : 20 phrases, dont 10 de test ; le score est très sensible à chaque erreur.
- **Phrases fictives**, rédigées pour être difficiles et générées par un LLM : elles ne représentent pas la fréquence
  réelle des tournures dans les comptes rendus (la négation simple « no X » y domine largement).
- **Annotation** : accord modéré (kappa 0,51) avant discussion, signe que la définition de la négation demande un
  guide précis ; les désaccords portaient sur la portée et les fausses négations, exactement là où les règles
  hésitent.
- **Règles** : faciles à expliquer (chaque décision affiche son déclencheur) et instantanées, mais fragiles hors des
  formes listées et des portées en nombre de mots.

**Conclusion** : les règles v0, écrites avant les phrases et jamais modifiées, classent correctement 18 des 20
mentions (10/10 en développement, 8/10 en test ; moyenne des deux F1 = 0,80 sur le test). Leurs deux erreurs
correspondent aux deux limites connues de NegEx : une négation qui porte sur un verbe d'exclusion, et une négation
éloignée de sa mention.